<a href="https://colab.research.google.com/github/tylerjohnbecker/Nonlinear_and_Data_Driven_Estimation/blob/main/BouncingBallNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Imports

In [123]:
import numpy as np
import matplotlib.pyplot as plt
import pandas
import math
import random
import copy
from scipy.integrate import odeint

In [124]:
import sys
import requests
import importlib

def import_local_or_github(package_name, function_name=None, directory=None, giturl=None):
    # Import functions directly from github
    # Important: note that we use raw.githubusercontent.com, not github.com

    try: # to find the file locally
        if directory is not None:
            if directory not in sys.path:
                sys.path.append(directory)

        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package, function_name)
            return function
        else:
            return package

    except: # get the file from github
        if giturl is None:
            giturl = 'https://raw.githubusercontent.com/florisvb/Nonlinear_and_Data_Driven_Estimation/main/Utility/' + str(package_name) + '.py'

        r = requests.get(giturl)
        print('Fetching from: ')
        print(r)

        # Store the file to the colab working directory
        with open(package_name+'.py', 'w') as f:
            f.write(r.text)
        f.close()

        # import the function we want from that file
        package = importlib.import_module(package_name)
        if function_name is not None:
            function = getattr(package , function_name)
            return function
        else:
            return package

plot_tme = import_local_or_github('plot_utility', 'plot_tme', directory='../Utility')
extended_kalman_filter = import_local_or_github('extended_kalman_filter', directory='../Utility')
unscented_kalman_filter = import_local_or_github('unscented_kalman_filter', directory='../Utility')

In [125]:
try:
    import casadi as ca
except:
    !pip install casadi
    import casadi as ca

try:
    import do_mpc
except:
    !pip install do_mpc
    import do_mpc

try:
    import pybounds
except:
    #!pip install pybounds
    !pip install git+https://github.com/vanbreugel-lab/pybounds
    import pybounds

#Linear Dynamics

In [126]:
g = -9.8
x0 = np.array([0, 0, random.random() * 8 + 2, 0, 0, random.random() - .5])
m = random.random() * 5
r = random.random() * 2
e = random.random()
mu = random.random()
Im = 0.4 * m * r ** 2

In [127]:
def estimate_damping_from_cor(e, m, stiffness):

    # Calculate damping ratio from coefficient of restitution
    ln_e = np.log(e)

    xi = np.sqrt(ln_e**2 / (np.pi**2 + ln_e**2))

    # For very low e, increase damping further
    correction = 1.0 + (0.3 - e) * 2.0

    xi = ca.if_else(e < .3, xi, xi * correction);

    # Calculate critical damping
    damping = 2 * xi * np.sqrt(m * stiffness)

    damping = ca.if_else(ca.fabs(e - 1.0) < 1e-6, 0.0, damping);

    return np.array([damping]).flatten()[0]

In [128]:
def f_in_air_linear(x_vec, u_vec, g, r, Im, m):

    # Extract state variables
    x = x_vec[0]
    x_dot = x_vec[1]
    z = x_vec[2]
    z_dot = x_vec[3]
    theta = x_vec[4]
    theta_dot = x_vec[5]

    # Extract control inputs
    f_x = u_vec[0]
    f_z = u_vec[1]
    tor = u_vec[2]

    # f0 component: drift dynamics (no controls)
    f0_contribution = np.array([ x_dot,
                                 0,
                                 z_dot,
                                 g,
                                 theta_dot,
                                 0])

    # f1 component: multiplied by control u1
    f_other_contribution = np.array([0,
                                     f_x / m,
                                     0,
                                     f_z / m,
                                     0,
                                     tor / Im])

    x_dot_vec = f0_contribution + f_other_contribution

    return x_dot_vec

In [129]:
def f_impact_linear(x_vec, u_vec, g, r, Im, m, e, mu, stiffness, damping):
    x_dot = x_vec[1]
    z_dot = x_vec[3]
    theta_dot = x_vec[5]
    z = x_vec[2]

    f_x, f_z, tau = u_vec[0], u_vec[1], u_vec[2]

    # Penetration depth (positive when in contact)
    penetration = r - z

    # Normal force (Hunt-Crossley contact model)
    # Only active when penetration > 0
    f_normal = stiffness * penetration - damping * penetration * z_dot
    f_normal = np.array([ca.if_else(f_normal > 0, f_normal, 0)]).flatten()[0]
    f_normal = np.array([ca.if_else(penetration > 0, f_normal, 0)]).flatten()[0]

    # Tangential velocity at contact point (bottom of ball)
    v_contact = x_dot - r * theta_dot

    # Friction force opposes contact point motion
    # If v_contact > 0 (sliding right), friction acts left (negative)
    # If v_contact < 0 (sliding left), friction acts right (positive)
    v_threshold = 0.1  # m/s - smoothing parameter
    friction_direction = np.tanh(v_contact / v_threshold)
    f_friction_ideal = -mu * f_normal * friction_direction

    # Cap friction force to prevent unrealistic accelerations
    max_friction_force = 1.0  # Newtons - tune based on your ball mass

    friction_sign = np.sign(f_friction_ideal)
    f_friction_ideal = friction_sign * f_friction_ideal

    f_friction = np.array([ca.if_else(f_friction_ideal > max_friction_force, max_friction_force, f_friction_ideal)]).flatten()[0] * friction_sign

    # f_friction = np.array([ca.if_else(f_friction_ideal > max_friction_force or f_friction_ideal < -max_friction_force, np.sign(f_friction_ideal) * max_friction_force, f_friction_ideal)]).flatten()[0]
    # f_friction = np.clip(f_friction_ideal, -max_friction_force, max_friction_force)

    # if v_contact > .5:
    #     print(f"v_contact: {v_contact} | rot: {r * theta_dot}")
    #     print(f"z: {z} | N: {f_normal } | fric: {f_friction} | dir: {friction_direction}")

    # Equations of motion
    x_dot_vec = np.array([
        x_dot,
        (f_x + f_friction) / m,
        z_dot,
        -g + (f_z + f_normal) / m,
        theta_dot,
        (tau - r * f_friction) / Im
    ])

    return x_dot_vec

In [130]:
def sigm_transition( z, r, width=.05):
    center = r + width
    return 1 / (1 + np.exp(-(z - center) / width))

In [131]:
def f_combined_linear(x_vec, u_vec, g=g, Im=Im, r=r, m=m, e=e, mu=mu, stiffness=100, transition_width=0.01):
    """
    Combined dynamics with smooth transition.
    """
    alpha = sigm_transition(x_vec[2], r, transition_width)

    f_air = f_in_air_linear(x_vec, u_vec, g, r, Im, m)
    f_cont = f_impact_linear(x_vec, u_vec, g, r, Im, m, e, mu, stiffness, estimate_damping_from_cor(e, m, stiffness))

    return alpha * f_air + (1 - alpha) * f_cont

In [132]:
def f_ode(x_vec, tsim, u_func, f):
    u_vec = u_func(x_vec, tsim)
    x_dot_vec = f(x_vec, u_vec)

    return x_dot_vec

# Basic Controllers

In [133]:
def u_go_right_slow(x_vec, tsim, target=1):

    right_diff = target - x_vec[1]

    return np.array([right_diff, 0, 0])

In [134]:
def u_go_right_fast(x_vec, tsim):
    return u_go_right_slow(x_vec, tsim, 10)

In [135]:
def u_go_left_slow(x_vec, tsim):
    return u_go_right_slow(x_vec, tsim, -1)

In [136]:
def u_go_left_fast(x_vec, tsim):
    return u_go_right_slow(x_vec, tsim, -10)

In [137]:
def u_bounce_slow(x_vec, tsim):
    return np.array([0, 7, 0])

In [138]:
def u_big_bounce(x_vec, tsim):
    return np.array([0, -10, 0])

In [139]:
def u_bigger_bounce(x_vec, tsim):
    if x_vec[3] < 0:
        return np.array([0, -100, 0])

    return np.array([0, 0, 0])

In [140]:
def u_spin_right_slow(x_vec, tsim, target=1):

    right_diff = target - x_vec[5]

    return np.array([0, 0, right_diff])

In [141]:
def u_crazy_backspin(x_vec, tsim):
    if x_vec[3] < 0:
        return u_spin_right_slow(x_vec, tsim, target=-100)

    return np.array([0, 0, 0])

In [142]:
def u_spin_left_slow(x_vec, tsim):
    return u_spin_right_slow(x_vec, tsim, -1)

In [143]:
def u_spin_right_fast(x_vec, tsim):
    return u_spin_right_slow(x_vec, tsim, 10)

In [144]:
def u_spin_left_fast(x_vec, tsim):
    return u_spin_right_slow(x_vec, tsim, 10)

In [145]:
def u_null(x_vec, tsim):
    return np.array([0, 0, 0])

## putting everything in a vector for randomly picking a controller

In [146]:
x_dot_controllers = [ u_go_right_slow, u_go_right_fast, u_go_left_slow, u_go_left_fast, u_null]
z_dot_controllers = [ u_bounce_slow, u_big_bounce, u_bigger_bounce, u_null ]
theta_dot_controllers = [ u_spin_right_slow, u_spin_left_slow, u_spin_right_fast, u_spin_left_fast, u_crazy_backspin, u_null ]

In [147]:
def get_random_controller():
    return lambda x_vec, tsim: random.choice(x_dot_controllers)(x_vec, tsim) + random.choice(z_dot_controllers)(x_vec, tsim) + random.choice(theta_dot_controllers)(x_vec, tsim)

# Measurement Function

In [148]:
def h_xyt(x_vec):
    return np.array([x_vec[0], x_vec[2], x_vec[5]])

# Generate Random Trajectories

In [149]:
# constants
g = -9.8
delta_t = .05

tsim = np.arange(0, 10, delta_t)

## Trajectory Utility Functions

In [150]:
def create_random_trajectory():

    # random variables
    x0 = np.array([0, 0, random.random() * 8 + 2, 0, 0, random.random() - .5])
    m = random.random() * 5
    r = random.random() * 2
    e = random.random()
    mu = random.random()
    Im = 0.4 * m * r ** 2

    # random controller selection
    u_rand_x = random.choice(x_dot_controllers)
    u_rand_z = random.choice(z_dot_controllers)
    u_rand_theta = random.choice(theta_dot_controllers)

    u_rand = lambda x_vec, tsim: u_rand_x(x_vec, tsim) + u_rand_z(x_vec, tsim) + u_rand_theta(x_vec, tsim)

    traj = np.array(odeint(f_ode, x0, tsim, args=(u_rand, f_combined_linear)))

    us =  [ u_rand(i, 0) for i in traj ]

    return [{'x0': x0, 'm': m, 'r': r, 'e': e, 'mu': mu, 'Im': Im}, traj, us]

In [151]:
def plot_trajectory(tsim, result):

    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.plot(result[:, 0], result[:, 2], '.')
    ax.set_xlabel('x pos')
    ax.set_ylabel('z pos')

    fig1 = plt.figure()
    ax1 = fig1.add_subplot(111)
    ax1.plot(tsim, result[:,1])
    ax1.set_xlabel('time')
    ax1.set_ylabel('x_dot')

    fig2 = plt.figure()
    ax2 = fig2.add_subplot(111)
    ax2.plot(tsim, result[:,3])
    ax2.set_xlabel('time')
    ax2.set_ylabel('z_dot')

    fig3 = plt.figure()
    ax3 = fig3.add_subplot(111)
    ax3.plot(tsim, result[:,5])
    ax3.set_xlabel('time')
    ax3.set_ylabel('theta_dot')

In [152]:
def cut_trajectory(trajectory, h_func):
    to_ret = []
    labels = []
    inputs = []
    params = []

    traj = []
    lab = []
    inp = []

    random_vars = trajectory[0]

    bouncing = False

    for index, x_vec in enumerate(trajectory[1]):
        if x_vec[2] < .7 + random_vars['r']:
            bouncing = True

            lab.append([random_vars['mu'], random_vars['e']])
            traj.append(h_func(x_vec))
            inp.append(trajectory[2][index])
        elif bouncing and not len(traj) == 0 and not len(lab) == 0:
            bouncing = False
            to_ret.append(traj)
            inputs.append(inp)
            labels.append(lab)
            params.append(random_vars)

            traj = []
            lab = []
            inp = []

    return to_ret, inputs, labels, params

## Create 1000 random trajectories

In [153]:
def loading_bar_print(cur_val, max_val, bar_size=30):

    proportion = cur_val / max_val

    num_eq = int(proportion * bar_size)

    print(f"\r {100 * (cur_val/max_val):.2f}%, {cur_val}/{max_val} || [", end="")
    print("="*(num_eq - 1) , end="")
    print(">", end="")
    print("-"*(bar_size - num_eq), end="")
    print(f"]", end="")

In [154]:
data = []
labels = []
inputs = []
params = []

num_trajectories = 20000

for i in range(num_trajectories):
    n_data, n_inputs, n_labels, n_params = cut_trajectory(create_random_trajectory(), h_xyt)

    if not n_data == [] and not n_labels == []:

        for index in range(len(n_data)):
            data.append(n_data[index])
            inputs.append(n_inputs[index])
            labels.append(n_labels[index])
            params.append(n_params[index])

    loading_bar_print(i + 1, num_trajectories, 50)

 0.09%, 18/20000 || [>--------------------------------------------------]

/tmp/ipykernel_1288849/2981269197.py:3: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-(z - center) / width))


 100.00%, 20000/20000 || [=================================================>]

# Neural Network Setup

## Organize the Data Into Frame Windows

In [155]:
window_size = 10

In [156]:
# fixed size stack class so that things are not super slow
class FixedSizeStack:
    class Node:
        def __init__(self, data, next):
            self.data = data
            self.next = next

    def __init__(self, max_size, filler):
        self.max_size = max_size
        self.top_ptr = None

        self.blank = filler

        self.size = 0

    def _get_back_ptr(self):
        cur = self.top_ptr

        while cur.next is not None:
            cur = cur.next

        return cur

    def push(self, data):
        if self.top_ptr is None:
            self.top_ptr = self.Node(data, None)
        else:
            new_node = self.Node(data, self.top_ptr)
            self.top_ptr = new_node

        self.size += 1

        if self.size > self.max_size:
            self._pop()

    def _pop(self):
        if self.top_ptr is None:
            return None
        else:
            cur = self.top_ptr

            while cur.next is not None and cur.next.next is not None:
                cur = cur.next

            cur.next = None

            self.size -= 1

            return cur.data

    def to_list(self):
        cur = self.top_ptr
        to_ret = []

        while cur is not None:
            to_ret.append(cur.data)
            cur = cur.next

        while len(to_ret) < self.max_size:
            to_ret.append(self.blank)

        # to_ret.reverse()

        return to_ret

In [157]:
def compute_velocity(init_val, final_val, delta_t=delta_t):
    assert delta_t != 0, "Delta t cannot be 0"
    return (final_val - init_val) / delta_t

In [210]:
def compute_confidence_labels(input_vector, window_size=window_size):
    """
    Compute confidence labels for μ and e estimation from flattened input vector.

    Args:
        input_vector: 1D array of shape (1 + window_size * 8,)
            Format: [radius, x_0, z_0, theta_0, x_dot_0, z_dot_0, fx_0, fz_0, tau_0,
                     x_1, z_1, theta_1, x_dot_1, z_dot_1, fx_1, fz_1, tau_1, ...]
        window_size: int, number of time frames (default 10)

    Returns:
        conf_mu: float in [0.1, 1.0]
        conf_e: float in [0.1, 1.0]
    """

    # Parse input vector
    radius = input_vector[0]

    # Extract time series data
    frame_data = input_vector[3:].reshape(window_size, 8)

    x = frame_data[:, 0]
    z = frame_data[:, 1]
    theta = frame_data[:, 2]
    x_dot = frame_data[:, 3]
    z_dot = frame_data[:, 4]
    fx = frame_data[:, 5]
    fz = frame_data[:, 6]
    tau = frame_data[:, 7]

    # Contact detection threshold
    contact_threshold = 0.1

    # Find contact events (z < threshold)
    in_contact = z < (contact_threshold + radius)

    # Find contact transitions
    contact_starts = np.where(np.diff(in_contact.astype(int)) == 1)[0] + 1
    contact_ends = np.where(np.diff(in_contact.astype(int)) == -1)[0] + 1

    # Handle edge cases
    if in_contact[0]:
        contact_starts = np.concatenate([[0], contact_starts])
    if in_contact[-1]:
        contact_ends = np.concatenate([contact_ends, [len(z) - 1]])

    n_contacts = min(len(contact_starts), len(contact_ends))

    if n_contacts == 0:
        return 0.1, 0.1

    contact_starts = contact_starts[:n_contacts]
    contact_ends = contact_ends[:n_contacts]

    # === Compute μ confidence factors ===
    conf_mu_factors = []

    for start, end in zip(contact_starts, contact_ends):
        start_idx = max(0, start - 1)
        end_idx = min(len(z) - 1, end)

        # 1. Normal impulse magnitude
        if start_idx < len(z_dot) and end_idx < len(z_dot):
            normal_impulse = abs(z_dot[end_idx] - z_dot[start_idx])
            conf_mu_factors.append(np.tanh(normal_impulse / 2.0))

        # 2. Tangential velocity at contact
        if start_idx < len(x_dot):
            tangential_vel = abs(x_dot[start_idx])
            conf_mu_factors.append(np.tanh(tangential_vel / 1.0))

        # 3. Angular velocity change (friction torque effect)
        # Note: using theta since theta_dot wasn't in original,
        # compute angular velocity from theta
        if start_idx > 0 and end_idx < len(theta):
            # Estimate theta_dot from theta differences
            theta_dot_start = theta[start_idx] - theta[start_idx - 1] if start_idx > 0 else 0
            theta_dot_end = theta[end_idx] - theta[end_idx - 1] if end_idx > 0 else 0
            theta_dot_change = abs(theta_dot_end - theta_dot_start)
            conf_mu_factors.append(np.tanh(theta_dot_change / 1.0))

    # === Compute e confidence factors ===
    conf_e_factors = []

    for start, end in zip(contact_starts, contact_ends):
        start_idx = max(0, start - 1)
        end_idx = min(len(z) - 1, end)

        # 1. Impact velocity magnitude
        if start_idx < len(z_dot):
            impact_velocity = abs(z_dot[start_idx])
            conf_e_factors.append(np.tanh(impact_velocity / 2.0))

        # 2. Rebound velocity
        if end_idx < len(z_dot):
            rebound_velocity = abs(z_dot[end_idx])
            conf_e_factors.append(np.tanh(rebound_velocity / 1.0))

        # 3. Perpendicularity of impact
        if start_idx < len(x_dot) and start_idx < len(z_dot):
            impact_angle = np.abs(np.arctan2(
                x_dot[start_idx],
                abs(z_dot[start_idx]) + 1e-6
            ))
            perpendicularity = 1.0 - (impact_angle / (np.pi / 2))
            conf_e_factors.append(max(0.0, perpendicularity))

    # Aggregate confidence scores
    conf_mu_base = np.mean(conf_mu_factors) if len(conf_mu_factors) > 0 else 0.1
    conf_e_base = np.mean(conf_e_factors) if len(conf_e_factors) > 0 else 0.1

    # Bonus for multiple contacts
    contact_bonus = min(0.2, n_contacts * 0.05)

    conf_mu = np.clip(conf_mu_base + contact_bonus, 0.1, 1.0)
    conf_e = np.clip(conf_e_base + contact_bonus, 0.1, 1.0)

    return [conf_mu, conf_e]

In [159]:
def to_windows(trajectories, inputs):
    to_ret = []

    filler = np.array([0,0,0,0,0,0,0,0])
    stack = FixedSizeStack(window_size, filler)

    prev_x = trajectories[0][0]
    prev_z = trajectories[0][2]

    for index in range(len(trajectories)):

        x_dot = compute_velocity(prev_x, trajectories[index][0])
        z_dot = compute_velocity(prev_z, trajectories[index][2])

        n_addition = np.hstack((trajectories[index], np.array([x_dot, z_dot]), inputs[index]))

        stack.push(n_addition)

        prev_x = trajectories[index][0]
        prev_z = trajectories[index][2]

        to_ret.append(stack.to_list())

    return to_ret

In [206]:
output_names = ['mu', 'e', 'mu_conf', 'e_conf']

# spaghetti to get the input names list correct
input_names = [['radius', 'mu_prior', 'e_prior']]
tmp = [ [f'x_{i}', f'z_{i}', f'theta_{i}', f'x_dot_{i}', f'z_dot_{i}', f'fx_{i}', f'fz_{i}', f'tau_{i}'] for i in range(window_size)]
tmp = sum(tmp, [])
input_names.append(tmp)
input_names = sum(input_names, [])

In [207]:
len(data)

76071

In [208]:
import pandas as pd

def make_data(trajectories, inputs, gt, params, input_names=input_names, output_names=output_names):
    x_data = []
    confidences = []

    for index in range(len(trajectories)):
        windows = to_windows(trajectories[index], inputs[index])

        confidences.append([])

        for window in windows:

            x_vec = [params[index]['r'], np.clip(params[index]['mu'] + np.random.normal(0, 0.1), 0, 1.0), np.clip(params[index]['e'] + np.random.normal(0, 0.05), 0, 1)]

            for i in window:
                x_vec.append(i[0])
                x_vec.append(i[1])
                x_vec.append(i[2])
                x_vec.append(i[3])
                x_vec.append(i[4])
                x_vec.append(i[5])
                x_vec.append(i[6])
                x_vec.append(i[7])

            conf_mu, conf_e = compute_confidence_labels(np.array(x_vec))
            confidences[-1].append([conf_mu, conf_e])
            x_data.append(x_vec)

    y_data = []

    for outer_index in range(len(gt)):
        for inner_index in range(len(gt[outer_index])):
            y_data.append([gt[outer_index][inner_index][0],
                            gt[outer_index][inner_index][1],
                            confidences[outer_index][inner_index][0],
                            confidences[outer_index][inner_index][1]])

    x_df = pd.DataFrame(x_data, columns=input_names)
    y_df = pd.DataFrame(y_data, columns=output_names)

    return x_df, y_df

In [211]:
x_df, y_df = make_data(data, inputs, labels, params)

In [212]:
x_df

,radius,mu_prior,e_prior,x_0,z_0,theta_0,x_dot_0,z_dot_0,fx_0,fz_0,...,fz_8,tau_8,x_9,z_9,theta_9,x_dot_9,z_dot_9,fx_9,fz_9,tau_9
0,1.558503,0.585852,0.097853,0.000000,2.156521,8.904218,0.000000,0.000000,0.000000,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
1,1.558503,0.931669,0.132263,0.000000,1.504427,9.088952,0.000000,3.694689,0.000000,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
2,1.558503,0.836653,0.191073,0.000001,0.800484,9.240457,0.000021,3.030090,0.000000,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
3,1.558503,0.850906,0.030305,0.001226,0.142736,9.232798,0.024496,-0.153178,0.000000,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
4,1.558503,0.657563,0.109257,0.005202,-0.233387,9.216208,0.079528,-0.331795,0.000000,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740516,0.887430,0.447655,0.387971,-2.343927,0.439963,0.513175,-0.696237,-1.464464,-0.321889,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
740517,0.887430,0.395056,0.539419,-2.377409,0.745767,0.519170,-0.669642,0.119887,-0.332986,-10.0,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
740518,0.887430,0.440165,0.426832,-2.411116,1.064974,0.594759,-0.674148,1.511779,-0.317183,-10.0,...,-10.0,0.029930,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000
740519,0.887430,0.451864,0.477663,-2.445688,1.336422,0.663077,-0.691433,1.366367,-0.300109,-10.0,...,-10.0,0.024884,-2.106806,1.369993,0.970070,0.000000,0.000000,-0.150057,-10.0,0.029930


In [213]:
n_input = len(input_names)
n_output = len(output_names)

# Try Fully Connected Neural Network

In [166]:
import tensorflow as tf
from tensorflow.python.client import device_lib
import keras
from IPython.display import display, Image

2025-12-05 11:35:06.964303: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-05 11:35:06.994408: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 11:35:07.148387: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-12-05 11:35:07.149109: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-05 11:35:07.711727: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not fin

In [214]:
# Define model architecture
# Try deeper/wider with residual connections
input_architecture = [
    {'core_input_dim': n_input},
]

core_architecture = [
    # Wider layers
    {'units': 256, 'activation': 'relu'},
    {'batch_norm': True},
    {'dropout': 0.2},

    {'units': 512, 'activation': 'relu'},
    {'batch_norm': True},
    {'dropout': 0.2},

    {'units': 512, 'activation': 'relu'},
    {'batch_norm': True},
    {'dropout': 0.2},

    {'units': 256, 'activation': 'relu'},
    {'batch_norm': True},
    {'dropout': 0.2},

    {'units': 128, 'activation': 'relu'},
    {'dropout': 0.1},

    {'units': 64, 'activation': 'relu'},
    {'units': n_output, 'activation': 'sigmoid'}
]

In [215]:
def build_model(input_architecture, core_architecture):
    model = keras.models.Sequential() # sequential model means that each layer has one set of inputs & one set of outputs (no recurrence)
    n_input = input_architecture[0]['core_input_dim']

    for i, layer in enumerate(core_architecture):
        if i == 0:
            model.add(keras.layers.Dense(layer['units'],
                            input_dim=n_input,
                            activation=layer['activation'])) # add a dense layer with 50 neurons & rectified linear unit (ReLU) activation function
        elif 'dropout' in layer:
            model.add(keras.layers.Dropout(layer['dropout']))
        elif 'batch_norm' in layer:
            model.add(keras.layers.BatchNormalization())
        else:
            model.add(keras.layers.Dense(layer['units'],
                            activation=layer['activation']))
    return model

In [216]:
def build_residual_model(n_input):
    """
    Network predicts delta_mu, delta_e (corrections to prior)
    """
    inputs = keras.Input(shape=(n_input,))

    # Extract prior estimates (indices 1 and 2)
    mu_prior = keras.layers.Lambda(lambda x: x[:, 1:2])(inputs)
    e_prior = keras.layers.Lambda(lambda x: x[:, 2:3])(inputs)

    # Process full input
    x = keras.layers.Dense(128, activation='relu')(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.Dense(256, activation='relu')(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.Dense(128, activation='relu')(x)
    x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.Dense(64, activation='relu')(x)

    # Predict residuals (tanh → [-1, 1] range for corrections)
    delta_mu = keras.layers.Dense(1, activation='tanh', name='delta_mu')(x)
    delta_e = keras.layers.Dense(1, activation='tanh', name='delta_e')(x)

    # Scale residuals (max correction ±0.3)
    delta_mu_scaled = keras.layers.Lambda(lambda x: x * 0.4)(delta_mu)
    delta_e_scaled = keras.layers.Lambda(lambda x: x * 0.3)(delta_e)

    # Add to prior
    mu_corrected = keras.layers.Add()([mu_prior, delta_mu_scaled])
    e_corrected = keras.layers.Add()([e_prior, delta_e_scaled])

    # Clip to valid range
    mu_out = keras.layers.Lambda(
        lambda x: tf.clip_by_value(x, 0.0, 1.0)
    )(mu_corrected)
    e_out = keras.layers.Lambda(
        lambda x: tf.clip_by_value(x, 0.0, 1.0)
    )(e_corrected)

    # Confidence outputs (unchanged)
    conf_mu = keras.layers.Dense(1, activation='sigmoid', name='conf_mu')(x)
    conf_e = keras.layers.Dense(1, activation='sigmoid', name='conf_e')(x)

    # Concatenate outputs
    outputs = keras.layers.Concatenate()([mu_out, e_out, conf_mu, conf_e])

    return keras.Model(inputs=inputs, outputs=outputs)

In [217]:
model = build_residual_model(n_input)
# model = build_model(input_architecture, core_architecture)

In [218]:
model.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_5 (InputLayer)           [(None, 83)]         0           []                               
                                                                                                  
 dense_34 (Dense)               (None, 128)          10752       ['input_5[0][0]']                
                                                                                                  
 batch_normalization_20 (BatchN  (None, 128)         512         ['dense_34[0][0]']               
 ormalization)                                                                                    
                                                                                                  
 dropout_26 (Dropout)           (None, 128)          0           ['batch_normalization_20[0]

In [219]:
def confidence_weighted_loss(y_true, y_pred):
    """
    Weighted loss that uses confidence labels

    y_true: [mu_true, e_true, conf_mu_true, conf_e_true]
    y_pred: [mu_pred, e_pred, conf_mu_pred, conf_e_pred]
    """
    # Extract components
    mu_true = y_true[:, 0]
    e_true = y_true[:, 1]
    conf_mu_true = y_true[:, 2]
    conf_e_true = y_true[:, 3]

    mu_pred = y_pred[:, 0]
    e_pred = y_pred[:, 1]
    conf_mu_pred = y_pred[:, 2]
    conf_e_pred = y_pred[:, 3]

    # Confidence-weighted parameter losses
    mu_loss = conf_mu_true * tf.square(mu_pred - mu_true)
    e_loss = conf_e_true * tf.square(e_pred - e_true)

    # Confidence prediction losses
    conf_mu_loss = tf.square(conf_mu_pred - conf_mu_true)
    conf_e_loss = tf.square(conf_e_pred - conf_e_true)

    # Calibration loss (penalize overconfident wrong predictions)
    mu_error = tf.abs(mu_pred - mu_true)
    e_error = tf.abs(e_pred - e_true)
    calibration_mu = conf_mu_pred * mu_error
    calibration_e = conf_e_pred * e_error

    # Combine
    total_loss = (
        tf.reduce_mean(mu_loss) +
        tf.reduce_mean(e_loss) +
        0.3 * tf.reduce_mean(conf_mu_loss) +
        0.3 * tf.reduce_mean(conf_e_loss) +
        0.2 * tf.reduce_mean(calibration_mu) +
        0.2 * tf.reduce_mean(calibration_e)
    )

    return total_loss

In [220]:
# Hyperparameters
config = {
    'learning_rate': 5e-4,
    'batch_size': 64,
    'epochs': 100,
    'validation_split': 0.2,
}

# Optimizer
optimizer = keras.optimizers.Adam(
    learning_rate=config['learning_rate']
)

# Compile model
model.compile(
    optimizer=optimizer,
    loss=confidence_weighted_loss,
    metrics=['mae']
)

# Callbacks
callbacks = [
    # Early stopping
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),

    # Learning rate reduction
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),

    # Stop if validation MAE plateaus
    keras.callbacks.EarlyStopping(
        monitor='val_mae',
        patience=15,
        mode='min',
        restore_best_weights=True
    ),

    # Model checkpoint
    keras.callbacks.ModelCheckpoint(
        'best_friction_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

In [221]:
model_data = model.fit(x_df, y_df, epochs=100, batch_size=32, validation_split=0.2, verbose=1,
                       callbacks=callbacks)

Epoch 1/100
18508/18513 [============================>.] - ETA: 0s - loss: 0.0216 - mae: 0.0725
Epoch 1: val_loss improved from inf to 0.01730, saving model to best_friction_model.keras
18513/18513 [==============================] - 69s 4ms/step - loss: 0.0216 - mae: 0.0725 - val_loss: 0.0173 - val_mae: 0.0596 - lr: 5.0000e-04
Epoch 2/100
18512/18513 [============================>.] - ETA: 0s - loss: 0.0171 - mae: 0.0592
Epoch 2: val_loss improved from 0.01730 to 0.01593, saving model to best_friction_model.keras
18513/18513 [==============================] - 81s 4ms/step - loss: 0.0171 - mae: 0.0592 - val_loss: 0.0159 - val_mae: 0.0530 - lr: 5.0000e-04
Epoch 3/100
18512/18513 [============================>.] - ETA: 0s - loss: 0.0162 - mae: 0.0557
Epoch 3: val_loss improved from 0.01593 to 0.01560, saving model to best_friction_model.keras
18513/18513 [==============================] - 78s 4ms/step - loss: 0.0162 - mae: 0.0557 - val_loss: 0.0156 - val_mae: 0.0529 - lr: 5.0000e-04
Epoch

# Save Everything to Local Files

In [223]:
import pickle

training_data = (x_df.to_numpy(), y_df.to_numpy())

with open("training.pkl", "wb") as file:
    pickle.dump(training_data, file)

In [222]:
model.save('friction_model.h5')